# **PHẦN 3: CHIA VÀ BIẾN ĐỔI DỮ LIỆU**
## **1. Định nghĩa vấn đề**
+ **Mô tả**:
   - Phát hiện và phân loại mã độc trên các thiết bị di động Android bằng tập dữ liệu Android Malware Detection.
   - Dữ liệu ban đầu là file chứa dữ liệu đã được xử lý và trích ra các đặc trưng (features.parquet).

+ **Mục tiêu**:
   - Trích ra các đặc trưng cần thiết cho việc huấn luyện mô hình

## **2. Chuẩn bị vấn đề**

### 2.1. Import các thư viện cần thiết

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow
import math
import itertools

from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 500)
pd.set_option("display.max_rows", 500)
RANDOM_STATE = 42

### 2.2. Lấy tập dữ liệu đã xử lý

In [5]:
dataFrame = pd.read_parquet("../data_processed/features.parquet")
dataFrame.head()

,Flow Duration,Total Fwd Packets,Total Length of Fwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Fwd IAT Total,Fwd IAT Std,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Min,Fwd Header Length,Bwd Packets/s,Min Packet Length,Down/Up Ratio,Init_Win_bytes_backward,min_seg_size_forward,Active Mean,Active Std,Idle Mean,Idle Std,Label,s_bit_0,s_bit_1,s_bit_2,s_bit_3,s_bit_4,s_bit_5,s_bit_6,s_bit_7,s_bit_8,s_bit_9,s_bit_10,s_bit_11,s_bit_12,s_bit_13,s_bit_14,s_bit_15,s_bit_16,s_bit_17,s_bit_18,s_bit_19,s_bit_20,s_bit_21,s_bit_22,s_bit_23,s_bit_24,s_bit_25,s_bit_26,s_bit_27,s_bit_28,s_bit_29,s_bit_30,s_bit_31,d_bit_0,d_bit_1,d_bit_2,d_bit_3,d_bit_4,d_bit_5,d_bit_6,d_bit_7,d_bit_8,d_bit_9,d_bit_10,d_bit_11,d_bit_12,d_bit_13,d_bit_14,d_bit_15,d_bit_16,d_bit_17,d_bit_18,d_bit_19,d_bit_20,d_bit_21,d_bit_22,d_bit_23,d_bit_24,d_bit_25,d_bit_26,d_bit_27,d_bit_28,d_bit_29,d_bit_30,d_bit_31,Source Port_Grp_System,Source Port_Grp_Registered,Source Port_Grp_Dynamic,Destination Port_Grp_System,Destination Port_Grp_Registered,Destination Port_Grp_Dynamic,Destination Port_Flag_DNS,Destination Port_Flag_Dynamic,Destination Port_Flag_HTTP,Destination Port_Flag_HTTPS,Destination Port_Flag_Other_Registered,Destination Port_Flag_Other_System,Destination Port_Flag_Proxy,Destination Port_Flag_SMTP,Hour_sin,Hour_cos,Fwd PSH Flags_0.0,Fwd PSH Flags_1.0,FIN Flag Count_0.0,FIN Flag Count_1.0,PSH Flag Count_0.0,PSH Flag Count_1.0,ACK Flag Count_0.0,ACK Flag Count_1.0,URG Flag Count_0.0,URG Flag Count_1.0,Protocol_0.0,Protocol_6.0,Protocol_17.0
0,37027,1,0.0,0.0,0.0,0.000,0.0,0.0,0.000000,0.000000,54.014638,3.702700e+04,0.000000e+00,0.0,0.000000e+00,0.0,0.000000,0.000000,0.0,32,27.007319,0.0,1,362.0,32.0,0.0,0.0,0.0,0.0,Android_Adware,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,1,0,1,0,1,1,0,0,1,1,0,1,1,0,0,1,0,0,0,0,0,1,1,0,1,1,0,0,1,0,1,0,0,0,1,1,0,0,0,0,0,1,0,0,0,0,-0.5,0.866025,1,0,1,0,1,0,0,1,0,1,0,1,0
1,36653,1,0.0,0.0,0.0,0.000,0.0,0.0,0.000000,0.000000,54.565793,3.665300e+04,0.000000e+00,0.0,0.000000e+00,0.0,0.000000,0.000000,0.0,32,27.282896,0.0,1,362.0,32.0,0.0,0.0,0.0,0.0,Android_Adware,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,1,0,1,0,1,1,0,0,1,1,0,1,1,0,0,1,0,0,0,0,0,1,1,0,1,1,0,0,1,0,1,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,-0.5,0.866025,1,0,1,0,1,0,0,1,0,1,0,1,0
2,534099,8,1011.0,581.0,0.0,126.375,1460.0,0.0,993.666667,24218.356522,37.446241,2.811047e+04,4.314810e+04,481340.0,6.237618e+04,487990.0,44362.727273,86342.042540,8.0,180,22.467745,0.0,1,63441.0,20.0,0.0,0.0,0.0,0.0,Android_Adware,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,1,0,0,0,0,0,1,1,1,1,1,1,1,1,0,1,0,0,1,1,1,1,0,1,0,1,0,0,0,1,0,0,0,0,1,1,0,0,0,0,0,1,0,0,0,0,-0.5,0.866025,1,0,1,0,0,1,1,0,1,0,0,1,0
3,9309,3,0.0,0.0,0.0,0.000,0.0,0.0,0.000000,0.000000,322.268772,4.654500e+03,5.137131e+03,9309.0,5.137131e+03,0.0,0.000000,0.000000,0.0,60,0.000000,0.0,0,-1.0,20.0,0.0,0.0,0.0,0.0,Android_Adware,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,1,0,0,0,0,0,1,1,1,1,1,1,1,1,0,1,0,0,1,1,1,1,0,1,0,1,0,0,0,1,0,0,0,0,1,1,0,0,0,0,0,1,0,0,0,0,-0.5,0.866025,1,0,1,0,1,0,0,1,1,0,0,1,0
4,19890496,8,430.0,218.0,0.0,53.750,1460.0,0.0,946.500000,307.131607,0.703854,1.530038e+06,5.377887e+06,19890496.0,7.314093e+06,410964.0,82192.800000,154845.683018,7.0,180,0.301652,0.0,0,64022.0,20.0,0.0,0.0,0.0,0.0,Android_Adware,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,1,0,0,0,0,0,1,1,1,1,1,1,1,1,0,1,0,0,1,1,1,1,0,1,0,1,0,0,0,1,0,0,0,0,1,1,0,0,0,0,0,1,0,0,0,0,-0.5,0.866025,1,0,1,0,0,1,1,0,1,0,0,1,0


In [6]:
dataFrame.describe()

,Flow Duration,Total Fwd Packets,Total Length of Fwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Fwd IAT Total,Fwd IAT Std,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Min,Fwd Header Length,Bwd Packets/s,Min Packet Length,Down/Up Ratio,Init_Win_bytes_backward,min_seg_size_forward,Active Mean,Active Std,Idle Mean,Idle Std,s_bit_0,s_bit_1,s_bit_2,s_bit_3,s_bit_4,s_bit_5,s_bit_6,s_bit_7,s_bit_8,s_bit_9,s_bit_10,s_bit_11,s_bit_12,s_bit_13,s_bit_14,s_bit_15,s_bit_16,s_bit_17,s_bit_18,s_bit_19,s_bit_20,s_bit_21,s_bit_22,s_bit_23,s_bit_24,s_bit_25,s_bit_26,s_bit_27,s_bit_28,s_bit_29,s_bit_30,s_bit_31,d_bit_0,d_bit_1,d_bit_2,d_bit_3,d_bit_4,d_bit_5,d_bit_6,d_bit_7,d_bit_8,d_bit_9,d_bit_10,d_bit_11,d_bit_12,d_bit_13,d_bit_14,d_bit_15,d_bit_16,d_bit_17,d_bit_18,d_bit_19,d_bit_20,d_bit_21,d_bit_22,d_bit_23,d_bit_24,d_bit_25,d_bit_26,d_bit_27,d_bit_28,d_bit_29,d_bit_30,d_bit_31,Source Port_Grp_System,Source Port_Grp_Registered,Source Port_Grp_Dynamic,Destination Port_Grp_System,Destination Port_Grp_Registered,Destination Port_Grp_Dynamic,Destination Port_Flag_DNS,Destination Port_Flag_Dynamic,Destination Port_Flag_HTTP,Destination Port_Flag_HTTPS,Destination Port_Flag_Other_Registered,Destination Port_Flag_Other_System,Destination Port_Flag_Proxy,Destination Port_Flag_SMTP,Hour_sin,Hour_cos,Fwd PSH Flags_0.0,Fwd PSH Flags_1.0,FIN Flag Count_0.0,FIN Flag Count_1.0,PSH Flag Count_0.0,PSH Flag Count_1.0,ACK Flag Count_0.0,ACK Flag Count_1.0,URG Flag Count_0.0,URG Flag Count_1.0,Protocol_0.0,Protocol_6.0,Protocol_17.0
count,3.556260e+05,355626.000000,3.556260e+05,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,3.556260e+05,3.556260e+05,3.556260e+05,3.556260e+05,3.556260e+05,3.556260e+05,3.556260e+05,3.556260e+05,3.556260e+05,3.556260e+05,3.556260e+05,355626.000000,355626.000000,355626.000000,355626.000000,3.556260e+05,3.556260e+05,3.556260e+05,3.556260e+05,3.556260e+05,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.00000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.00000,355626.000000,355626.000000,355626.00000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,3.556260e+05,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000,355626.000000
mean,1.092987e+07,7.357412,6.791997e+02,212.001524,12.520283,59.644118,320.406978,23.057701,168.537248,8.399002e+04,5.494626e+03,3.175841e+06,2.426832e+06,7.540020e+06,1.934147e+06,5.632490e+06,9.461668e+05,1.407311e+06,3.426236e+05,-7.601343e+05,814.702967,8.413744,0.556461,2032.585548,-2.831850e+04,1.628143e+05,2.195225e+04,4.025724e+06,3.182527e+05,0.048320,0.047401,0.060035,0.040565,0.960217,0.060513,0.941562,0.035998,0.055435,0.062495,0.923855,0.050857,0.949264,0.05393,0.940651,0.064298,0.038864,0.037022,0.026854,0.031601,0.056191,0.042784,0.054214,0.043338,0.887834,0.546240,0.106154,0.882599,0.095775,0.387221,0.

## **3. Thực hiện vấn đề**

In [9]:
mapping = {'Android_Adware': 0,
            'Android_Scareware': 1,
            'Android_SMS_Malware': 2,
            'Benign': 3
          }
dataFrame['Label'] = dataFrame['Label'].map(mapping)
dataFrame.head()

,Flow Duration,Total Fwd Packets,Total Length of Fwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Fwd IAT Total,Fwd IAT Std,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Min,Fwd Header Length,Bwd Packets/s,Min Packet Length,Down/Up Ratio,Init_Win_bytes_backward,min_seg_size_forward,Active Mean,Active Std,Idle Mean,Idle Std,Label,s_bit_0,s_bit_1,s_bit_2,s_bit_3,s_bit_4,s_bit_5,s_bit_6,s_bit_7,s_bit_8,s_bit_9,s_bit_10,s_bit_11,s_bit_12,s_bit_13,s_bit_14,s_bit_15,s_bit_16,s_bit_17,s_bit_18,s_bit_19,s_bit_20,s_bit_21,s_bit_22,s_bit_23,s_bit_24,s_bit_25,s_bit_26,s_bit_27,s_bit_28,s_bit_29,s_bit_30,s_bit_31,d_bit_0,d_bit_1,d_bit_2,d_bit_3,d_bit_4,d_bit_5,d_bit_6,d_bit_7,d_bit_8,d_bit_9,d_bit_10,d_bit_11,d_bit_12,d_bit_13,d_bit_14,d_bit_15,d_bit_16,d_bit_17,d_bit_18,d_bit_19,d_bit_20,d_bit_21,d_bit_22,d_bit_23,d_bit_24,d_bit_25,d_bit_26,d_bit_27,d_bit_28,d_bit_29,d_bit_30,d_bit_31,Source Port_Grp_System,Source Port_Grp_Registered,Source Port_Grp_Dynamic,Destination Port_Grp_System,Destination Port_Grp_Registered,Destination Port_Grp_Dynamic,Destination Port_Flag_DNS,Destination Port_Flag_Dynamic,Destination Port_Flag_HTTP,Destination Port_Flag_HTTPS,Destination Port_Flag_Other_Registered,Destination Port_Flag_Other_System,Destination Port_Flag_Proxy,Destination Port_Flag_SMTP,Hour_sin,Hour_cos,Fwd PSH Flags_0.0,Fwd PSH Flags_1.0,FIN Flag Count_0.0,FIN Flag Count_1.0,PSH Flag Count_0.0,PSH Flag Count_1.0,ACK Flag Count_0.0,ACK Flag Count_1.0,URG Flag Count_0.0,URG Flag Count_1.0,Protocol_0.0,Protocol_6.0,Protocol_17.0
0,37027,1,0.0,0.0,0.0,0.000,0.0,0.0,0.000000,0.000000,54.014638,3.702700e+04,0.000000e+00,0.0,0.000000e+00,0.0,0.000000,0.000000,0.0,32,27.007319,0.0,1,362.0,32.0,0.0,0.0,0.0,0.0,0,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,1,0,1,0,1,1,0,0,1,1,0,1,1,0,0,1,0,0,0,0,0,1,1,0,1,1,0,0,1,0,1,0,0,0,1,1,0,0,0,0,0,1,0,0,0,0,-0.5,0.866025,1,0,1,0,1,0,0,1,0,1,0,1,0
1,36653,1,0.0,0.0,0.0,0.000,0.0,0.0,0.000000,0.000000,54.565793,3.665300e+04,0.000000e+00,0.0,0.000000e+00,0.0,0.000000,0.000000,0.0,32,27.282896,0.0,1,362.0,32.0,0.0,0.0,0.0,0.0,0,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,1,0,1,0,1,1,0,0,1,1,0,1,1,0,0,1,0,0,0,0,0,1,1,0,1,1,0,0,1,0,1,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,-0.5,0.866025,1,0,1,0,1,0,0,1,0,1,0,1,0
2,534099,8,1011.0,581.0,0.0,126.375,1460.0,0.0,993.666667,24218.356522,37.446241,2.811047e+04,4.314810e+04,481340.0,6.237618e+04,487990.0,44362.727273,86342.042540,8.0,180,22.467745,0.0,1,63441.0,20.0,0.0,0.0,0.0,0.0,0,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,1,0,0,0,0,0,1,1,1,1,1,1,1,1,0,1,0,0,1,1,1,1,0,1,0,1,0,0,0,1,0,0,0,0,1,1,0,0,0,0,0,1,0,0,0,0,-0.5,0.866025,1,0,1,0,0,1,1,0,1,0,0,1,0
3,9309,3,0.0,0.0,0.0,0.000,0.0,0.0,0.000000,0.000000,322.268772,4.654500e+03,5.137131e+03,9309.0,5.137131e+03,0.0,0.000000,0.000000,0.0,60,0.000000,0.0,0,-1.0,20.0,0.0,0.0,0.0,0.0,0,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,1,0,0,0,0,0,1,1,1,1,1,1,1,1,0,1,0,0,1,1,1,1,0,1,0,1,0,0,0,1,0,0,0,0,1,1,0,0,0,0,0,1,0,0,0,0,-0.5,0.866025,1,0,1,0,1,0,0,1,1,0,0,1,0
4,19890496,8,430.0,218.0,0.0,53.750,1460.0,0.0,946.500000,307.131607,0.703854,1.530038e+06,5.377887e+06,19890496.0,7.314093e+06,410964.0,82192.800000,154845.683018,7.0,180,0.301652,0.0,0,64022.0,20.0,0.0,0.0,0.0,0.0,0,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,1,0,0,0,0,0,1,1,1,1,1,1,1,1,0,1,0,0,1,1,1,1,0,1,0,1,0,0,0,1,0,0,0,0,1,1,0,0,0,0,0,1,0,0,0,0,-0.5,0.866025,1,0,1,0,0,1,1,0,1,0,0,1,0


In [14]:
X = dataFrame.drop(columns="Label")
y = dataFrame["Label"]

x_train, x_temp, y_train, y_temp = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

x_val, x_test, y_val, y_test = train_test_split(
    x_temp, y_temp, 
    test_size=0.50, 
    random_state=42, 
    stratify=y_temp
)

In [16]:
num_cols = x_test.select_dtypes(include="number")

filtered = num_cols.loc[:, 
    (num_cols.nunique() < 100)
]

filtered.head()

,Fwd Packet Length Min,Min Packet Length,Down/Up Ratio,min_seg_size_forward,s_bit_0,s_bit_1,s_bit_2,s_bit_3,s_bit_4,s_bit_5,s_bit_6,s_bit_7,s_bit_8,s_bit_9,s_bit_10,s_bit_11,s_bit_12,s_bit_13,s_bit_14,s_bit_15,s_bit_16,s_bit_17,s_bit_18,s_bit_19,s_bit_20,s_bit_21,s_bit_22,s_bit_23,s_bit_24,s_bit_25,s_bit_26,s_bit_27,s_bit_28,s_bit_29,s_bit_30,s_bit_31,d_bit_0,d_bit_1,d_bit_2,d_bit_3,d_bit_4,d_bit_5,d_bit_6,d_bit_7,d_bit_8,d_bit_9,d_bit_10,d_bit_11,d_bit_12,d_bit_13,d_bit_14,d_bit_15,d_bit_16,d_bit_17,d_bit_18,d_bit_19,d_bit_20,d_bit_21,d_bit_22,d_bit_23,d_bit_24,d_bit_25,d_bit_26,d_bit_27,d_bit_28,d_bit_29,d_bit_30,d_bit_31,Source Port_Grp_System,Source Port_Grp_Registered,Source Port_Grp_Dynamic,Destination Port_Grp_System,Destination Port_Grp_Registered,Destination Port_Grp_Dynamic,Destination Port_Flag_DNS,Destination Port_Flag_Dynamic,Destination Port_Flag_HTTP,Destination Port_Flag_HTTPS,Destination Port_Flag_Other_Registered,Destination Port_Flag_Other_System,Destination Port_Flag_Proxy,Destination Port_Flag_SMTP,Hour_sin,Hour_cos,Fwd PSH Flags_0.0,Fwd PSH Flags_1.0,FIN Flag Count_0.0,FIN Flag Count_1.0,PSH Flag Count_0.0,PSH Flag Count_1.0,ACK Flag Count_0.0,ACK Flag Count_1.0,URG Flag Count_0.0,URG Flag Count_1.0,Protocol_0.0,Protocol_6.0,Protocol_17.0
159186,0.0,0.0,0,32.0,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,1,1,1,1,0,1,0,1,1,0,0,1,1,0,1,1,0,0,1,0,0,0,0,1,0,1,0,1,1,1,0,0,1,0,0,0,0,1,1,0,0,0,0,0,1,0,0,0,0,1.000000e+00,6.123234e-17,1,0,1,0,0,1,1,0,1,0,0,1,0
205409,0.0,0.0,3,32.0,0,0,1,1,0,1,1,0,1,1,1,0,0,1,0,1,1,0,0,0,1,1,1,1,0,0,1,1,1,0,1,1,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,1,0,0,0,1,0,0,0,0,0,1,0,0,0,8.660254e-01,-5.000000e-01,1,0,1,0,1,0,0,1,1,0,0,1,0
297273,0.0,0.0,0,20.0,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,1,0,0,1,1,1,1,0,0,1,0,0,0,1,1,1,1,1,0,0,0,1,1,1,1,0,1,0,0,0,1,1,1,1,0,0,1,1,0,0,0,0,1,0,0,0,0,0,-2.449294e-16,1.000000e+00,1,0,1,0,1,0,0,1,0,1,0,1,0
100019,0.0,0.0,0,20.0,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,1,0,1,1,0,1,1,1,1,1,1,0,1,0,0,0,1,1,1,0,0,1,1,1,1,0,1,0,1,1,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,8.660254e-01,-5.000000e-01,1,0,1,0,1,0,0,1,0,1,0,1,0
49851,0.0,0.0,0,32.0,0,0,0,0,1,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,1,1,0,1,0,0,1,1,0,0,0,0,0,0,1,1,0,1,0,0,0,1,1,1,0,0,1,1,0,0,0,1,0,0,0,0,1,1,0,0,0,0,0,1,0,0,0,0,-5.000000e-01,8.660254e-01,1,0,1,0,1,0,0,1,1,0,0,1,0


In [ ]:
cols = ["Fwd PSH Flags", "FIN Flag Count", "PSH Flag Count", "ACK Flag Count", "URG Flag Count", "Protocol"]

encoder = OneHotEncoder(sparse_output=False, dtype='int')
encoded_array = encoder.fit_transform(dataFrame[cols])
encoded_df = pd.DataFrame(
    encoded_array, 
    columns=encoder.get_feature_names_out(cols),
    index=dataFrame.index
)
dataFrame = pd.concat([dataFrame.drop(columns=cols), encoded_df], axis=1)
dataFrame.head()